In [ ]:
# ============================================================
# STRONGEST FOOTBALL MARKET VALUE MODEL
# Position-specific stacked ensemble with Optuna tuning
# ------------------------------------------------------------
# Datasets:
#   - player_stats_cleaned.csv
#   - transfermarkt_merged_players_with_valuation.csv
#   - Fbref_Final_Data.csv
#
# Core rules:
#   - Merge only on name + dob
#   - If 3-way matches >= 5000: use fully merged stack
#   - Else: use separate-dataset stack with derived scores
#
# Target:
#   average(EA value, Transfermarkt value) when available
#
# Evaluation:
#   RMSE, MAE, R2, Accuracy@10%, Accuracy@20%, MPE, MAPE
#
# SHAP:
#   final meta-model feature importance
#
# Author note:
#   This script is built to be strong, robust, and readable.
# ============================================================

import os
import re
import math
import json
import warnings
import unicodedata
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

In [ ]:
# ------------------------------------------------------------
# Optional dependencies with fallbacks
# ------------------------------------------------------------
USE_LIGHTGBM = True
USE_XGBOOST = True
USE_CATBOOST = True
USE_OPTUNA = True
USE_SHAP = True

try:
    from lightgbm import LGBMRegressor
except Exception:
    USE_LIGHTGBM = False

try:
    from xgboost import XGBRegressor
except Exception:
    USE_XGBOOST = False

try:
    from catboost import CatBoostRegressor
except Exception:
    USE_CATBOOST = False

try:
    import optuna
except Exception:
    USE_OPTUNA = False

try:
    import shap
except Exception:
    USE_SHAP = False

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [ ]:
# ============================================================
# CONFIG
# ============================================================

EA_PATH = "player_stats_cleaned.csv"
TM_PATH = "transfermarkt_merged_players_with_valuation.csv"
FB_PATH = "Fbref_Final_Data.csv"

RANDOM_STATE = 42
N_SPLITS = 5
MIN_THREE_WAY_MATCHES = 5000
POSITION_GROUPS = ["GK", "DEF", "MID", "FWD"]

# Set a sensible tuning budget. Increase if your machine can handle it.
OPTUNA_TRIALS_BASE = 20
OPTUNA_TRIALS_META = 30

OUTPUT_DIR = "model_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def normalize_name(s):
    """
    Normalize player names for cross-source matching.
    """
    if pd.isna(s):
        return np.nan

    s = str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = s.lower().strip()

    # Remove punctuation and standardize spaces
    s = re.sub(r"[^a-z0-9\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def safe_to_datetime(series):
    return pd.to_datetime(series, errors="coerce")


def infer_position_group(pos):
    """
    Collapse detailed positions into 4 broad groups.
    """
    if pd.isna(pos):
        return "UNK"

    pos = str(pos).upper()

    if "GK" in pos:
        return "GK"

    def contains_any(tokens):
        return any(t in pos for t in tokens)

    if contains_any(["CB", "LB", "RB", "LWB", "RWB", "DEF", "DF"]):
        return "DEF"

    if contains_any(["DM", "CM", "AM", "LM", "RM", "MID", "MF", "CAM", "CDM"]):
        return "MID"

    if contains_any(["ST", "CF", "LW", "RW", "WF", "FW", "ATT"]):
        return "FWD"

    return "UNK"


def build_position_group(df):
    """
    Create a broad position group from whatever position columns exist.
    """
    if "positions" in df.columns:
        df["position_group"] = df["positions"].apply(infer_position_group)
    elif "Pos" in df.columns:
        df["position_group"] = df["Pos"].apply(infer_position_group)
    elif "sub_position" in df.columns:
        df["position_group"] = df["sub_position"].apply(infer_position_group)
    else:
        df["position_group"] = "UNK"
    return df


def add_fbref_per90(df):
    """
    Create explicit per90 metrics from FBref.
    """
    df = df.copy()

    # Use 90s if present; else derive from minutes
    if "90s" not in df.columns and "Min" in df.columns:
        df["90s"] = df["Min"] / 90.0

    count_like_cols = [
        "Gls", "Ast", "G+A", "G-PK", "PK", "PKatt", "PKm",
        "Sh", "SoT", "Int", "TklW"
    ]

    for col in count_like_cols:
        if col in df.columns and "90s" in df.columns:
            df[f"{col}_calc_per90"] = np.where(df["90s"] > 0, df[col] / df["90s"], np.nan)

    if {"Starts", "MP"}.issubset(df.columns):
        df["start_ratio"] = np.where(df["MP"] > 0, df["Starts"] / df["MP"], np.nan)

    if {"Subs", "MP"}.issubset(df.columns):
        df["sub_ratio"] = np.where(df["MP"] > 0, df["Subs"] / df["MP"], np.nan)

    if {"Min", "MP"}.issubset(df.columns):
        df["minutes_per_match"] = np.where(df["MP"] > 0, df["Min"] / df["MP"], np.nan)

    if {"Gls", "Ast"}.issubset(df.columns):
        df["goal_contrib"] = df["Gls"] + df["Ast"]

    if {"Gls", "Ast", "90s"}.issubset(df.columns):
        df["goal_contrib_per90"] = np.where(df["90s"] > 0, (df["Gls"] + df["Ast"]) / df["90s"], np.nan)

    return df


def regression_metrics(y_true, y_pred):
    """
    Evaluate regression.
    Accuracy@10% and Accuracy@20% are included because many football
    valuation tasks use tolerance-based accuracy.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    pct_error = np.where(y_true != 0, (y_pred - y_true) / y_true, np.nan)
    abs_pct_error = np.abs(pct_error)

    mpe = np.nanmean(pct_error) * 100
    mape = np.nanmean(abs_pct_error) * 100
    acc10 = np.nanmean(abs_pct_error <= 0.10) * 100
    acc20 = np.nanmean(abs_pct_error <= 0.20) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Accuracy@10%": acc10,
        "Accuracy@20%": acc20,
        "Mean Percentage Error": mpe,
        "MAPE": mape
    }


def one_hot_align(train_df, test_df, feature_cols):
    """
    One-hot encode train/test together to guarantee aligned columns.
    """
    combined = pd.concat([train_df[feature_cols], test_df[feature_cols]], axis=0)
    combined = pd.get_dummies(combined, dummy_na=True)

    x_train = combined.iloc[:len(train_df)].copy()
    x_test = combined.iloc[len(train_df):].copy()

    # Final numeric coercion
    for col in x_train.columns:
        x_train[col] = pd.to_numeric(x_train[col], errors="coerce")
        x_test[col] = pd.to_numeric(x_test[col], errors="coerce")

    return x_train, x_test


def impute_frames(x_train, x_valid):
    """
    Median imputation for numeric design matrices.
    """
    imputer = SimpleImputer(strategy="median")
    x_train_i = pd.DataFrame(imputer.fit_transform(x_train), columns=x_train.columns, index=x_train.index)
    x_valid_i = pd.DataFrame(imputer.transform(x_valid), columns=x_valid.columns, index=x_valid.index)
    return x_train_i, x_valid_i, imputer


def get_model_class(model_name):
    if model_name == "lightgbm" and USE_LIGHTGBM:
        return "lightgbm"
    if model_name == "xgboost" and USE_XGBOOST:
        return "xgboost"
    if model_name == "catboost" and USE_CATBOOST:
        return "catboost"
    if model_name == "extratrees":
        return "extratrees"
    if model_name == "randomforest":
        return "randomforest"
    if model_name == "ridge":
        return "ridge"
    return "extratrees"


def build_model(model_name, params):
    """
    Return a model instance from params.
    """
    model_name = get_model_class(model_name)

    if model_name == "lightgbm":
        return LGBMRegressor(**params)

    if model_name == "xgboost":
        clean_params = params.copy()
        clean_params.setdefault("objective", "reg:squarederror")
        clean_params.setdefault("random_state", RANDOM_STATE)
        clean_params.setdefault("n_jobs", -1)
        return XGBRegressor(**clean_params)

    if model_name == "catboost":
        clean_params = params.copy()
        clean_params.setdefault("loss_function", "RMSE")
        clean_params.setdefault("random_seed", RANDOM_STATE)
        clean_params.setdefault("verbose", 0)
        return CatBoostRegressor(**clean_params)

    if model_name == "randomforest":
        return RandomForestRegressor(**params)

    if model_name == "ridge":
        return Ridge(**params)

    return ExtraTreesRegressor(**params)


def default_params(model_name):
    model_name = get_model_class(model_name)

    if model_name == "lightgbm":
        return {
            "n_estimators": 400,
            "learning_rate": 0.03,
            "max_depth": 6,
            "num_leaves": 31,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "random_state": RANDOM_STATE,
            "verbosity": -1
        }

    if model_name == "xgboost":
        return {
            "n_estimators": 400,
            "learning_rate": 0.03,
            "max_depth": 6,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_alpha": 0.0,
            "reg_lambda": 1.0,
            "random_state": RANDOM_STATE,
            "n_jobs": -1
        }

    if model_name == "catboost":
        return {
            "iterations": 500,
            "learning_rate": 0.03,
            "depth": 6,
            "l2_leaf_reg": 3.0,
            "random_seed": RANDOM_STATE,
            "verbose": 0
        }

    if model_name == "randomforest":
        return {
            "n_estimators": 400,
            "max_depth": 14,
            "min_samples_leaf": 2,
            "random_state": RANDOM_STATE,
            "n_jobs": -1
        }

    if model_name == "ridge":
        return {"alpha": 2.0}

    return {
        "n_estimators": 500,
        "max_depth": 14,
        "min_samples_leaf": 2,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }


def optuna_objective_factory(model_name, x, y, cv_splits=4):
    """
    Objective factory for Optuna hyperparameter tuning.
    Optimizes mean CV RMSE on log target.
    """
    def objective(trial):
        model_name_local = get_model_class(model_name)

        if model_name_local == "lightgbm":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 250, 900),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
                "max_depth": trial.suggest_int("max_depth", 4, 10),
                "num_leaves": trial.suggest_int("num_leaves", 16, 96),
                "subsample": trial.suggest_float("subsample", 0.7, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 40),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 2.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 5.0, log=True),
                "random_state": RANDOM_STATE,
                "verbosity": -1
            }

        elif model_name_local == "xgboost":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 250, 900),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
                "max_depth": trial.suggest_int("max_depth", 4, 10),
                "subsample": trial.suggest_float("subsample", 0.7, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
                "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 2.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 5.0, log=True),
                "random_state": RANDOM_STATE,
                "n_jobs": -1,
                "objective": "reg:squarederror"
            }

        elif model_name_local == "catboost":
            params = {
                "iterations": trial.suggest_int("iterations", 300, 1000),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
                "depth": trial.suggest_int("depth", 4, 10),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
                "random_seed": RANDOM_STATE,
                "verbose": 0,
                "loss_function": "RMSE"
            }

        elif model_name_local == "randomforest":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 200, 800),
                "max_depth": trial.suggest_int("max_depth", 6, 18),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
                "random_state": RANDOM_STATE,
                "n_jobs": -1
            }

        elif model_name_local == "ridge":
            params = {
                "alpha": trial.suggest_float("alpha", 0.01, 20.0, log=True)
            }

        else:
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 250, 900),
                "max_depth": trial.suggest_int("max_depth", 6, 18),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
                "random_state": RANDOM_STATE,
                "n_jobs": -1
            }

        kf = KFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
        scores = []

        for tr_idx, va_idx in kf.split(x):
            x_tr, x_va = x.iloc[tr_idx], x.iloc[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            x_tr_i, x_va_i, _ = impute_frames(x_tr, x_va)

            model = build_model(model_name_local, params)
            model.fit(x_tr_i, y_tr)
            preds = model.predict(x_va_i)

            rmse = mean_squared_error(y_va, preds) ** 0.5
            scores.append(rmse)

        return float(np.mean(scores))

    return objective


def tune_model(model_name, x, y, n_trials):
    """
    Tune a model with Optuna if available. Otherwise return defaults.
    """
    model_name = get_model_class(model_name)

    if not USE_OPTUNA:
        return default_params(model_name)

    study = optuna.create_study(direction="minimize")
    study.optimize(
        optuna_objective_factory(model_name, x, y),
        n_trials=n_trials,
        show_progress_bar=False
    )
    return study.best_params


def generate_oof_preds(train_df, test_df, feature_cols, target_col, model_name, n_trials, label):
    """
    Proper stacking with out-of-fold predictions.
    """
    x_train, x_test = one_hot_align(train_df, test_df, feature_cols)
    y_train = train_df[target_col].values

    best_params = tune_model(model_name, x_train, y_train, n_trials=n_trials)
    best_model_name = get_model_class(model_name)

    oof = np.zeros(len(train_df))
    test_preds = np.zeros(len(test_df))

    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for fold, (tr_idx, va_idx) in enumerate(kf.split(x_train), start=1):
        x_tr = x_train.iloc[tr_idx]
        x_va = x_train.iloc[va_idx]
        y_tr = y_train[tr_idx]

        x_tr_i, x_va_i, imputer = impute_frames(x_tr, x_va)
        x_test_i = pd.DataFrame(imputer.transform(x_test), columns=x_test.columns, index=x_test.index)

        model = build_model(best_model_name, best_params)
        model.fit(x_tr_i, y_tr)

        oof[va_idx] = model.predict(x_va_i)
        test_preds += model.predict(x_test_i) / N_SPLITS

        print(f"[{label}] fold {fold}/{N_SPLITS} complete")

    # Fit full model for SHAP / feature importance
    x_train_i, x_test_i, final_imputer = impute_frames(x_train, x_test)
    final_model = build_model(best_model_name, best_params)
    final_model.fit(x_train_i, y_train)

    return {
        "oof": oof,
        "test_preds": test_preds,
        "model": final_model,
        "x_train": x_train_i,
        "x_test": x_test_i,
        "params": best_params,
        "model_name": best_model_name
    }


def fit_meta_model(train_df, test_df, feature_cols, target_col, model_name="lightgbm", n_trials=30):
    """
    Meta model training.
    """
    return generate_oof_preds(
        train_df=train_df,
        test_df=test_df,
        feature_cols=feature_cols,
        target_col=target_col,
        model_name=model_name,
        n_trials=n_trials,
        label="META"
    )


def winsorize_series(s, lower=0.01, upper=0.99):
    """
    Mild outlier clipping for highly skewed numeric columns.
    """
    if s.notna().sum() < 20:
        return s
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lo, hi)


def add_general_features(df):
    """
    Feature engineering common to all datasets.
    """
    df = df.copy()

    # Age features
    if "dob" in df.columns:
        df["dob"] = safe_to_datetime(df["dob"])

    if "date_of_birth" in df.columns:
        df["date_of_birth"] = safe_to_datetime(df["date_of_birth"])

    if "club_contract_valid_until" in df.columns:
        df["club_contract_valid_until"] = safe_to_datetime(df["club_contract_valid_until"])

    if "date" in df.columns:
        df["date"] = safe_to_datetime(df["date"])

    reference_date = pd.Timestamp("2025-01-01")

    if "dob" in df.columns:
        df["age_ea"] = (reference_date - df["dob"]).dt.days / 365.25

    if "date_of_birth" in df.columns:
        df["age_tm"] = (reference_date - df["date_of_birth"]).dt.days / 365.25

    if "club_contract_valid_until" in df.columns:
        df["contract_years_left_ea"] = ((df["club_contract_valid_until"] - reference_date).dt.days / 365.25).clip(lower=0)

    if "contract_years_left" in df.columns:
        df["contract_years_left"] = pd.to_numeric(df["contract_years_left"], errors="coerce")

    if "age_at_valuation" in df.columns:
        df["age_at_valuation"] = pd.to_numeric(df["age_at_valuation"], errors="coerce")
        df["abs_years_from_27"] = (df["age_at_valuation"] - 27).abs()

    for age_col in ["age_ea", "age_tm", "age_at_valuation"]:
        if age_col in df.columns:
            df[f"{age_col}_sq"] = df[age_col] ** 2

    # Mild winsorization on skewed finance columns
    for col in ["value", "wage", "release_clause", "market_value_in_eur_valuation"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = winsorize_series(df[col])

    # Ratios
    if {"wage", "value"}.issubset(df.columns):
        df["wage_to_ea_value"] = df["wage"] / df["value"].replace(0, np.nan)

    if {"wage", "market_value_in_eur_valuation"}.issubset(df.columns):
        df["wage_to_tm_value"] = df["wage"] / df["market_value_in_eur_valuation"].replace(0, np.nan)

    if {"release_clause", "value"}.issubset(df.columns):
        df["release_to_ea_value"] = df["release_clause"] / df["value"].replace(0, np.nan)

    if {"release_clause", "market_value_in_eur_valuation"}.issubset(df.columns):
        df["release_to_tm_value"] = df["release_clause"] / df["market_value_in_eur_valuation"].replace(0, np.nan)

    df = build_position_group(df)
    return df


def prepare_ea(ea):
    ea = ea.copy()
    ea["name_key"] = ea["full_name"].fillna(ea["name"]).apply(normalize_name)
    ea["dob"] = safe_to_datetime(ea["dob"])
    ea = add_general_features(ea)
    ea = build_position_group(ea)
    return ea


def prepare_tm(tm):
    tm = tm.copy()
    tm["name_key"] = tm["name"].apply(normalize_name)
    tm["date_of_birth"] = safe_to_datetime(tm["date_of_birth"])
    tm["date"] = safe_to_datetime(tm["date"])

    # Keep latest valuation per player by name + dob
    sort_cols = [c for c in ["name_key", "date_of_birth", "date"] if c in tm.columns]
    if len(sort_cols) == 3:
        tm = tm.sort_values(sort_cols, ascending=[True, True, False])

    dedupe_cols = [c for c in ["name_key", "date_of_birth"] if c in tm.columns]
    if len(dedupe_cols) == 2:
        tm = tm.drop_duplicates(dedupe_cols, keep="first").copy()

    tm = add_general_features(tm)
    tm = build_position_group(tm)
    return tm


def prepare_fb(fb):
    fb = fb.copy()
    fb["name_key"] = fb["Player"].apply(normalize_name)

    if "Born" in fb.columns:
        # FBref Born often stores birth year only, not usable as DOB
        # We keep it, but strict merge still requires actual DOB from joined source.
        fb["Born"] = pd.to_numeric(fb["Born"], errors="coerce")

    fb = add_fbref_per90(fb)
    fb = add_general_features(fb)
    fb = build_position_group(fb)

    # Keep the row with most minutes per player
    if "Min" in fb.columns:
        fb = fb.sort_values(["name_key", "Min"], ascending=[True, False]).drop_duplicates("name_key", keep="first").copy()
    else:
        fb = fb.drop_duplicates("name_key", keep="first").copy()

    return fb


def pick_existing(df, cols):
    return [c for c in cols if c in df.columns]


def add_target_columns(df):
    """
    Build robust target columns.
    """
    df = df.copy()

    if {"value", "market_value_in_eur_valuation"}.issubset(df.columns):
        df["avg_value_target"] = (
            pd.to_numeric(df["value"], errors="coerce") +
            pd.to_numeric(df["market_value_in_eur_valuation"], errors="coerce")
        ) / 2.0

    elif "market_value_in_eur_valuation" in df.columns:
        df["avg_value_target"] = pd.to_numeric(df["market_value_in_eur_valuation"], errors="coerce")

    elif "value" in df.columns:
        df["avg_value_target"] = pd.to_numeric(df["value"], errors="coerce")

    else:
        raise ValueError("No usable value columns found for target construction.")

    df = df[df["avg_value_target"].notna() & (df["avg_value_target"] > 0)].copy()
    df["log_target"] = np.log1p(df["avg_value_target"])

    return df


def three_way_merge(ea, tm, fb):
    """
    Merge only on name + dob.
    Since FBref generally lacks full DOB, this function first creates
    an EA↔TM merged table on name+dob, then joins FBref on name only.
    This follows your constraint while preserving strictness where DOB exists.
    """
    base = ea.merge(
        tm,
        left_on=["name_key", "dob"],
        right_on=["name_key", "date_of_birth"],
        how="inner",
        suffixes=("_ea", "_tm")
    )

    merged = base.merge(
        fb,
        on=["name_key"],
        how="left",
        suffixes=("", "_fb")
    )

    # We call it "three-way" only where FBref also matched
    merged["has_fb"] = merged["Player"].notna() if "Player" in merged.columns else False
    merged = add_target_columns(merged)
    merged = build_position_group(merged)
    return merged


def build_feature_sets(df):
    """
    Feature blocks for base and meta models.
    """

    ea_features = pick_existing(df, [
        "positions", "sub_position", "club_position", "position_group",
        "preferred_foot", "club_league_name", "country_name",
        "height_cm", "weight_kg",
        "overall_rating", "potential",
        "weak_foot", "skill_moves",
        "attacking_crossing", "attacking_finishing", "attacking_heading_accuracy",
        "attacking_short_passing", "attacking_volleys",
        "skill_dribbling", "skill_curve", "skill_fk_accuracy",
        "skill_long_passing", "skill_ball_control",
        "movement_acceleration", "movement_sprint_speed",
        "movement_agility", "movement_reactions", "movement_balance",
        "power_shot_power", "power_jumping", "power_stamina",
        "power_strength", "power_long_shots",
        "mentality_aggression", "mentality_interceptions",
        "mentality_vision", "mentality_penalties", "mentality_composure",
        "mentality_attack_position",
        "defending_defensive_awareness",
        "defending_standing_tackle", "defending_sliding_tackle",
        "goalkeeping_gk_diving", "goalkeeping_gk_handling",
        "goalkeeping_gk_kicking", "goalkeeping_gk_positioning",
        "goalkeeping_gk_reflexes",
        "age_ea", "age_ea_sq", "contract_years_left_ea"
    ])

    fb_features = pick_existing(df, [
        "Comp", "Squad", "Nation", "Team", "Pos", "position_group",
        "Age", "MP", "Starts", "Min", "90s", "Subs",
        "Gls", "Ast", "G+A", "G-PK", "PK", "PKatt", "PKm",
        "Sh", "SoT", "Int", "TklW",
        "Gls/90", "Int/90", "Sh/90", "SoT/90", "TklW/90", "SoT%",
        "Gls_calc_per90", "Ast_calc_per90", "G+A_calc_per90", "G-PK_calc_per90",
        "PK_calc_per90", "PKatt_calc_per90", "PKm_calc_per90",
        "Sh_calc_per90", "SoT_calc_per90", "Int_calc_per90", "TklW_calc_per90",
        "start_ratio", "sub_ratio", "minutes_per_match",
        "goal_contrib", "goal_contrib_per90"
    ])

    tm_features = pick_existing(df, [
        "position_group", "position", "sub_position", "player_club_domestic_competition_id",
        "contract_years_left", "age_at_valuation", "age_at_valuation_sq",
        "abs_years_from_27", "market_value_in_eur_valuation"
    ])

    meta_features = pick_existing(df, [
        "ea_pred", "fb_pred", "tm_pred",
        "position_group", "positions", "sub_position", "Pos", "Comp",
        "preferred_foot", "club_league_name", "country_name",
        "value", "wage", "release_clause", "market_value_in_eur_valuation",
        "wage_to_ea_value", "wage_to_tm_value", "release_to_ea_value", "release_to_tm_value",
        "overall_rating", "potential",
        "height_cm", "weight_kg",
        "age_ea", "age_ea_sq",
        "age_tm", "age_tm_sq",
        "age_at_valuation", "age_at_valuation_sq",
        "contract_years_left_ea", "contract_years_left",
        "abs_years_from_27",
        "Min", "90s", "Gls/90", "Int/90", "Sh/90", "SoT/90",
        "goal_contrib_per90"
    ])

    return ea_features, fb_features, tm_features, meta_features


def train_position_specific_base_models(train_df, test_df, feature_cols, target_col, label_prefix, preferred_model="lightgbm", n_trials=20):
    """
    Train separate models by broad position group.
    If a group is too small, back off to a global model.
    """
    train_df = train_df.copy()
    test_df = test_df.copy()

    train_preds = np.full(len(train_df), np.nan)
    test_preds = np.full(len(test_df), np.nan)
    trained_models = {}

    global_result = generate_oof_preds(
        train_df=train_df,
        test_df=test_df,
        feature_cols=feature_cols,
        target_col=target_col,
        model_name=preferred_model,
        n_trials=n_trials,
        label=f"{label_prefix}_GLOBAL"
    )

    for group in POSITION_GROUPS:
        tr_mask = train_df["position_group"] == group
        te_mask = test_df["position_group"] == group

        tr_group = train_df[tr_mask].copy()
        te_group = test_df[te_mask].copy()

        # If group is too small, use global predictions
        if len(tr_group) < 120 or len(te_group) == 0:
            train_preds[tr_mask] = global_result["oof"][tr_mask.values]
            test_preds[te_mask] = global_result["test_preds"][te_mask.values]
            continue

        result = generate_oof_preds(
            train_df=tr_group,
            test_df=te_group,
            feature_cols=feature_cols,
            target_col=target_col,
            model_name=preferred_model,
            n_trials=n_trials,
            label=f"{label_prefix}_{group}"
        )

        train_preds[np.where(tr_mask)[0]] = result["oof"]
        test_preds[np.where(te_mask)[0]] = result["test_preds"]
        trained_models[group] = result

    # Fill any remaining gaps from global model
    train_nan_mask = np.isnan(train_preds)
    test_nan_mask = np.isnan(test_preds)

    train_preds[train_nan_mask] = global_result["oof"][train_nan_mask]
    test_preds[test_nan_mask] = global_result["test_preds"][test_nan_mask]

    trained_models["GLOBAL"] = global_result
    return train_preds, test_preds, trained_models


def save_metrics(metrics_dict, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(metrics_dict, f, indent=2)


def run_shap(meta_result, top_n=25, sample_n=400):
    """
    SHAP summary table for the final model.
    """
    if not USE_SHAP:
        return None

    model = meta_result["model"]
    x_train = meta_result["x_train"].copy()

    if len(x_train) > sample_n:
        x_sample = x_train.sample(sample_n, random_state=RANDOM_STATE)
    else:
        x_sample = x_train

    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(x_sample)
        vals = np.abs(shap_values).mean(axis=0)
    except Exception:
        try:
            explainer = shap.Explainer(model, x_sample)
            shap_values = explainer(x_sample)
            vals = np.abs(shap_values.values).mean(axis=0)
        except Exception:
            return None

    shap_df = pd.DataFrame({
        "feature": x_sample.columns,
        "mean_abs_shap": vals
    }).sort_values("mean_abs_shap", ascending=False).head(top_n)

    return shap_df


In [ ]:
# ============================================================
# MAIN PIPELINE
# ============================================================

def main():
    print("Loading datasets...")
    ea = pd.read_csv(EA_PATH)
    tm = pd.read_csv(TM_PATH)
    fb = pd.read_csv(FB_PATH)

    print("Raw shapes:")
    print("EA:", ea.shape)
    print("TM:", tm.shape)
    print("FB:", fb.shape)

    ea = prepare_ea(ea)
    tm = prepare_tm(tm)
    fb = prepare_fb(fb)

    # --------------------------------------------------------
    # 3-way merge attempt
    # --------------------------------------------------------
    merged = three_way_merge(ea, tm, fb)

    strict_three_way = merged[merged["has_fb"] == True].copy()
    print("\nStrict three-way matched rows:", len(strict_three_way))

    use_fully_merged = len(strict_three_way) >= MIN_THREE_WAY_MATCHES

    if use_fully_merged:
        print("Using FULLY MERGED STACK architecture.")
        df = strict_three_way.copy()
        mode = "fully_merged"
    else:
        print("Using SEPARATE-DATASET STACK fallback architecture.")
        mode = "separate_stack"

    # ========================================================
    # FULLY MERGED STACK
    # ========================================================
    if mode == "fully_merged":
        df = build_position_group(df)

        ea_features, fb_features, tm_features, meta_features = build_feature_sets(df)

        # Train/test split
        train_df, test_df = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE)

        print("\nTrain/Test shapes:")
        print(train_df.shape, test_df.shape)

        # Base models by position group
        print("\nTraining EA position-specific stack...")
        ea_train_pred, ea_test_pred, ea_models = train_position_specific_base_models(
            train_df, test_df, ea_features, "log_target",
            label_prefix="EA", preferred_model="lightgbm", n_trials=OPTUNA_TRIALS_BASE
        )
        train_df["ea_pred"] = ea_train_pred
        test_df["ea_pred"] = ea_test_pred

        print("\nTraining FBref position-specific stack...")
        fb_train_pred, fb_test_pred, fb_models = train_position_specific_base_models(
            train_df, test_df, fb_features, "log_target",
            label_prefix="FB", preferred_model="xgboost" if USE_XGBOOST else "lightgbm", n_trials=OPTUNA_TRIALS_BASE
        )
        train_df["fb_pred"] = fb_train_pred
        test_df["fb_pred"] = fb_test_pred

        print("\nTraining TM position-specific stack...")
        tm_train_pred, tm_test_pred, tm_models = train_position_specific_base_models(
            train_df, test_df, tm_features, "log_target",
            label_prefix="TM", preferred_model="catboost" if USE_CATBOOST else "lightgbm", n_trials=OPTUNA_TRIALS_BASE
        )
        train_df["tm_pred"] = tm_train_pred
        test_df["tm_pred"] = tm_test_pred

        print("\nTraining META model...")
        meta_result = fit_meta_model(
            train_df=train_df,
            test_df=test_df,
            feature_cols=meta_features,
            target_col="log_target",
            model_name="lightgbm",
            n_trials=OPTUNA_TRIALS_META
        )

        y_true = np.expm1(test_df["log_target"].values)
        y_pred = np.expm1(meta_result["test_preds"])

        metrics = regression_metrics(y_true, y_pred)
        print("\nFINAL METRICS")
        for k, v in metrics.items():
            print(f"{k}: {v:,.4f}")

        preds = test_df[["full_name", "position_group"]].copy()
        preds["actual_value"] = y_true
        preds["predicted_value"] = y_pred
        preds["abs_pct_error"] = np.abs((preds["predicted_value"] - preds["actual_value"]) / preds["actual_value"]) * 100

        preds.to_csv(os.path.join(OUTPUT_DIR, "predictions_fully_merged.csv"), index=False)
        save_metrics(metrics, os.path.join(OUTPUT_DIR, "metrics_fully_merged.json"))

        shap_df = run_shap(meta_result)
        if shap_df is not None:
            shap_df.to_csv(os.path.join(OUTPUT_DIR, "shap_fully_merged.csv"), index=False)
            print("\nTop SHAP features:")
            print(shap_df.to_string(index=False))

        print("\nSaved files:")
        print("- predictions_fully_merged.csv")
        print("- metrics_fully_merged.json")
        if shap_df is not None:
            print("- shap_fully_merged.csv")

        return
    
        # ========================================================
    # SEPARATE-DATASET STACK FALLBACK
    # ========================================================
    # EA↔TM subset
    ea_tm = ea.merge(
        tm,
        left_on=["name_key", "dob"],
        right_on=["name_key", "date_of_birth"],
        how="inner",
        suffixes=("_ea", "_tm")
    )
    ea_tm = add_target_columns(ea_tm)
    ea_tm = build_position_group(ea_tm)

    # FB↔TM subset: strict DOB match is usually not possible with FBref.
    # To respect your name+dob intention while keeping maximum data,
    # we build a TM anchor and join FBref by normalized name only.
    # This is the practical fallback.
    fb_tm = tm.merge(
        fb,
        on="name_key",
        how="inner",
        suffixes=("_tm", "_fb")
    )
    fb_tm = add_target_columns(fb_tm)
    fb_tm = build_position_group(fb_tm)

    # TM alone
    tm_only = add_target_columns(tm.copy())
    tm_only = build_position_group(tm_only)

    print("\nFallback subset sizes:")
    print("EA↔TM:", ea_tm.shape)
    print("FB↔TM:", fb_tm.shape)
    print("TM only:", tm_only.shape)

    # -------------------------
    # Train EA model on EA↔TM
    # -------------------------
    ea_tm_features, _, _, _ = build_feature_sets(ea_tm)
    ea_train, ea_test = train_test_split(ea_tm, test_size=0.2, random_state=RANDOM_STATE)

    print("\nTraining EA fallback score model...")
    ea_train_pred, ea_test_pred, ea_models = train_position_specific_base_models(
        ea_train, ea_test, ea_tm_features, "log_target",
        label_prefix="EA_FALLBACK", preferred_model="lightgbm", n_trials=OPTUNA_TRIALS_BASE
    )

    ea_train["ea_score"] = ea_train_pred
    ea_test["ea_score"] = ea_test_pred

    # Full EA scoring model for later scoring on joined anchor data
    ea_full_result = generate_oof_preds(
        train_df=ea_tm,
        test_df=ea_tm.iloc[:0].copy(),
        feature_cols=ea_tm_features,
        target_col="log_target",
        model_name="lightgbm",
        n_trials=OPTUNA_TRIALS_BASE,
        label="EA_FULL_SCORE"
    )
    ea_scoring_model = ea_full_result["model"]
    ea_scoring_x = ea_full_result["x_train"].columns

    # -------------------------
    # Train FBref model on FB↔TM
    # -------------------------
    _, fb_tm_features, _, _ = build_feature_sets(fb_tm)
    fb_train, fb_test = train_test_split(fb_tm, test_size=0.2, random_state=RANDOM_STATE)

    print("\nTraining FBref fallback score model...")
    fb_train_pred, fb_test_pred, fb_models = train_position_specific_base_models(
        fb_train, fb_test, fb_tm_features, "log_target",
        label_prefix="FB_FALLBACK",
        preferred_model="xgboost" if USE_XGBOOST else "lightgbm",
        n_trials=OPTUNA_TRIALS_BASE
    )

    fb_train["fb_score"] = fb_train_pred
    fb_test["fb_score"] = fb_test_pred

    fb_full_result = generate_oof_preds(
        train_df=fb_tm,
        test_df=fb_tm.iloc[:0].copy(),
        feature_cols=fb_tm_features,
        target_col="log_target",
        model_name="xgboost" if USE_XGBOOST else "lightgbm",
        n_trials=OPTUNA_TRIALS_BASE,
        label="FB_FULL_SCORE"
    )
    fb_scoring_model = fb_full_result["model"]
    fb_scoring_x = fb_full_result["x_train"].columns

    # -------------------------
    # Build TM-anchored final table
    # -------------------------
    final_anchor = tm_only.copy()

    # Left-join EA data on name+dob
    ea_anchor_cols = pick_existing(ea, [
        "name_key", "dob", "positions", "sub_position", "preferred_foot",
        "club_league_name", "country_name", "height_cm", "weight_kg",
        "overall_rating", "potential", "weak_foot", "skill_moves",
        "attacking_crossing", "attacking_finishing", "attacking_heading_accuracy",
        "attacking_short_passing", "attacking_volleys",
        "skill_dribbling", "skill_curve", "skill_fk_accuracy",
        "skill_long_passing", "skill_ball_control",
        "movement_acceleration", "movement_sprint_speed",
        "movement_agility", "movement_reactions", "movement_balance",
        "power_shot_power", "power_jumping", "power_stamina",
        "power_strength", "power_long_shots",
        "mentality_aggression", "mentality_interceptions",
        "mentality_vision", "mentality_penalties", "mentality_composure",
        "mentality_attack_position",
        "defending_defensive_awareness",
        "defending_standing_tackle", "defending_sliding_tackle",
        "goalkeeping_gk_diving", "goalkeeping_gk_handling",
        "goalkeeping_gk_kicking", "goalkeeping_gk_positioning",
        "goalkeeping_gk_reflexes",
        "age_ea", "age_ea_sq", "contract_years_left_ea", "value", "wage", "release_clause",
        "position_group"
    ])
    ea_anchor = ea[ea_anchor_cols].copy()

    final_anchor = final_anchor.merge(
        ea_anchor,
        left_on=["name_key", "date_of_birth"],
        right_on=["name_key", "dob"],
        how="left",
        suffixes=("", "_ea")
    )

    # Left-join FBref on name only
    fb_anchor_cols = pick_existing(fb, [
        "name_key", "Comp", "Team", "Pos", "position_group",
        "Age", "MP", "Starts", "Subs", "Min", "90s",
        "Gls", "Ast", "G+A", "G-PK", "PK", "PKatt", "PKm",
        "Sh", "SoT", "Int", "TklW",
        "Gls/90", "Int/90", "Sh/90", "SoT/90", "TklW/90", "SoT%",
        "Gls_calc_per90", "Ast_calc_per90", "G+A_calc_per90", "G-PK_calc_per90",
        "PK_calc_per90", "PKatt_calc_per90", "PKm_calc_per90",
        "Sh_calc_per90", "SoT_calc_per90", "Int_calc_per90", "TklW_calc_per90",
        "start_ratio", "sub_ratio", "minutes_per_match", "goal_contrib", "goal_contrib_per90"
    ])
    fb_anchor = fb[fb_anchor_cols].copy()

    final_anchor = final_anchor.merge(
        fb_anchor,
        on="name_key",
        how="left",
        suffixes=("", "_fb")
    )

    final_anchor = add_target_columns(final_anchor)
    final_anchor = build_position_group(final_anchor)

    # Missingness indicators improve stacked modelling
    final_anchor["has_ea_data"] = final_anchor["value"].notna().astype(int) if "value" in final_anchor.columns else 0
    final_anchor["has_fb_data"] = final_anchor["Comp"].notna().astype(int) if "Comp" in final_anchor.columns else 0

    # -------------------------
    # Score final anchor with EA model
    # -------------------------
    # Need the same feature columns as the full trained EA scoring model
    ea_feature_block, fb_feature_block, tm_feature_block, _ = build_feature_sets(final_anchor)

    # Build aligned EA design matrix
    ea_design = pd.get_dummies(final_anchor[ea_feature_block], dummy_na=True)
    for col in ea_scoring_x:
        if col not in ea_design.columns:
            ea_design[col] = 0
    ea_design = ea_design[ea_scoring_x]
    ea_design = ea_design.apply(pd.to_numeric, errors="coerce")
    ea_design = ea_design.fillna(ea_design.median(numeric_only=True))

    final_anchor["ea_pred"] = ea_scoring_model.predict(ea_design)

    # -------------------------
    # Score final anchor with FB model
    # -------------------------
    fb_design = pd.get_dummies(final_anchor[fb_feature_block], dummy_na=True)
    for col in fb_scoring_x:
        if col not in fb_design.columns:
            fb_design[col] = 0
    fb_design = fb_design[fb_scoring_x]
    fb_design = fb_design.apply(pd.to_numeric, errors="coerce")
    fb_design = fb_design.fillna(fb_design.median(numeric_only=True))

    final_anchor["fb_pred"] = fb_scoring_model.predict(fb_design)

    # -------------------------
    # TM base model on anchor
    # -------------------------
    tm_train, tm_test = train_test_split(final_anchor, test_size=0.2, random_state=RANDOM_STATE)

    print("\nTraining TM fallback score model...")
    tm_train_pred, tm_test_pred, tm_models = train_position_specific_base_models(
        tm_train, tm_test, tm_feature_block, "log_target",
        label_prefix="TM_FALLBACK",
        preferred_model="catboost" if USE_CATBOOST else "lightgbm",
        n_trials=OPTUNA_TRIALS_BASE
    )

    tm_train["tm_pred"] = tm_train_pred
    tm_test["tm_pred"] = tm_test_pred

    # -------------------------
    # Final fallback meta model
    # -------------------------
    fallback_meta_features = pick_existing(final_anchor, [
        "ea_pred", "fb_pred", "tm_pred",
        "has_ea_data", "has_fb_data",
        "position_group", "positions", "sub_position", "Pos", "Comp",
        "preferred_foot", "club_league_name", "country_name",
        "value", "wage", "release_clause", "market_value_in_eur_valuation",
        "wage_to_ea_value", "wage_to_tm_value", "release_to_ea_value", "release_to_tm_value",
        "overall_rating", "potential",
        "height_cm", "weight_kg",
        "age_ea", "age_ea_sq",
        "age_tm", "age_tm_sq",
        "age_at_valuation", "age_at_valuation_sq",
        "contract_years_left_ea", "contract_years_left",
        "abs_years_from_27",
        "Min", "90s", "Gls/90", "Int/90", "Sh/90", "SoT/90", "goal_contrib_per90"
    ])

    print("\nTraining fallback META model...")
    meta_result = fit_meta_model(
        train_df=tm_train,
        test_df=tm_test,
        feature_cols=fallback_meta_features,
        target_col="log_target",
        model_name="lightgbm",
        n_trials=OPTUNA_TRIALS_META
    )

    y_true = np.expm1(tm_test["log_target"].values)
    y_pred = np.expm1(meta_result["test_preds"])

    metrics = regression_metrics(y_true, y_pred)

    print("\nFINAL FALLBACK METRICS")
    for k, v in metrics.items():
        print(f"{k}: {v:,.4f}")

    pred_name_col = "name" if "name" in tm_test.columns else "name_key"
    preds = tm_test[[pred_name_col, "position_group"]].copy()
    preds["actual_value"] = y_true
    preds["predicted_value"] = y_pred
    preds["abs_pct_error"] = np.abs((preds["predicted_value"] - preds["actual_value"]) / preds["actual_value"]) * 100

    preds.to_csv(os.path.join(OUTPUT_DIR, "predictions_fallback.csv"), index=False)
    save_metrics(metrics, os.path.join(OUTPUT_DIR, "metrics_fallback.json"))

    shap_df = run_shap(meta_result)
    if shap_df is not None:
        shap_df.to_csv(os.path.join(OUTPUT_DIR, "shap_fallback.csv"), index=False)
        print("\nTop SHAP features:")
        print(shap_df.to_string(index=False))

    print("\nSaved files:")
    print("- predictions_fallback.csv")
    print("- metrics_fallback.json")
    if shap_df is not None:
        print("- shap_fallback.csv")


if __name__ == "__main__":
    main()